# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [255]:
%load_ext dotenv
%dotenv ../05_src/.secrets

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [256]:
# -----------------------------
# Load Libraries
# -----------------------------
from langchain_community.document_loaders import PyPDFLoader
import os
from openai import OpenAI
from pydantic import BaseModel
from IPython.display import display, Markdown
from deepeval.metrics import SummarizationMetric, GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams


In [257]:
# -----------------------------
# Load PDF File
# -----------------------------

document_folder = "./documents/"
file_name = "managing_oneself.pdf"
file_path = os.path.join(document_folder, file_name)
loader = PyPDFLoader(file_path)
docs = loader.load()

print(len(docs))

13


In [258]:
# -----------------------------
# Read Page Contents
# -----------------------------

document_text = ""
for page in docs:
    document_text += page.page_content + "\n"

print(document_text)

www.hbr.org
B
 
EST  
 
OF  HBR 1999
 
Managing Oneself
 
by Peter F . Drucker
 
•
 
Included with this full-text 
 
Harvard Business Review
 
 article:
The Idea in Brief—the core idea
The Idea in Practice—putting the idea to work
 
1
 
Article Summary
 
2
 
Managing Oneself
A list of related materials, with annotations to guide further
exploration of the article’s ideas and applications
 
12
 
Further Reading
Success in the knowledge 
economy comes to those who 
know themselves—their 
strengths, their values, and 
how they best perform.
 
Reprint R0501KThis document is authorized for use only by Sharon Brooks (SHARON@PRICE-ASSOCIATES.COM). Copying or posting is an infringement of copyright. Please contact 
customerservice@harvardbusiness.org or 800-988-0886 for additional copies.
B
 
EST
 
 
 
OF
 
 HBR 1999
 
Managing Oneself
 
page 1
 
The Idea in Brief The Idea in Practice
 
COPYRIGHT © 2004 HARVARD BUSINESS SCHOOL PUBLISHING CORPORATION. ALL RIGHTS RESERVED.
 
We live in an age of

## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [259]:
# -----------------------------
# Create OpenAI Client
# -----------------------------

client = OpenAI(base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1', 
                api_key='any value',
                default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')})


In [260]:
# -----------------------------
# Define Variables
# -----------------------------
# gpt_model = "gpt-4o-mini"
gpt_model = "gpt-4o"
summary_tone = "Formal Academic Writing"
number_of_tokens = 1000
context = document_text

# -----------------------------
# Define the Pydantic model
# -----------------------------
class ArticleSummary(BaseModel):
    Author: str
    Title: str
    Relevance: str
    Summary: str
    Tone: str
    InputTokens: int
    OutputTokens: int

In [261]:
# ---------------------------------------------------
# Developer instructions (kept separate)
# ---------------------------------------------------
developer_instructions = f"""
You are an assistant designed to summarize documents and return structured outputs.

Follow these rules:

1. Output MUST be a Pydantic BaseModel object with fields:
   - Author
   - Title
   - Relevance
   - Summary
   - Tone
   - InputTokens
   - OutputTokens
3. Summary must:
   - Be no longer than {number_of_tokens} tokens
   - Use a clearly identifiable tone: {summary_tone}
4. Relevance must be a short paragraph explaining why the article matters for an AI professional.
5. Use the provided document context to generate the summary.
6. Do NOT include anything outside the BaseModel fields.
"""


In [262]:
# ---------------------------------------------------
# User prompt (kept separate)
# ---------------------------------------------------
user_prompt_template = f"""
Summarize the following document using the specified tone.

Tone: Formal Academic Writing

Document:
{context}

Return ONLY a JSON object matching the Pydantic model.
"""


In [263]:
# ---------------------------------------------------
# Inject the PDF text dynamically
# ---------------------------------------------------
user_prompt = user_prompt_template.format(context=document_text)  # <-- your extracted PDF text
print(user_prompt)


Summarize the following document using the specified tone.

Tone: Formal Academic Writing

Document:
www.hbr.org
B
 
EST  
 
OF  HBR 1999
 
Managing Oneself
 
by Peter F . Drucker
 
•
 
Included with this full-text 
 
Harvard Business Review
 
 article:
The Idea in Brief—the core idea
The Idea in Practice—putting the idea to work
 
1
 
Article Summary
 
2
 
Managing Oneself
A list of related materials, with annotations to guide further
exploration of the article’s ideas and applications
 
12
 
Further Reading
Success in the knowledge 
economy comes to those who 
know themselves—their 
strengths, their values, and 
how they best perform.
 
Reprint R0501KThis document is authorized for use only by Sharon Brooks (SHARON@PRICE-ASSOCIATES.COM). Copying or posting is an infringement of copyright. Please contact 
customerservice@harvardbusiness.org or 800-988-0886 for additional copies.
B
 
EST
 
 
 
OF
 
 HBR 1999
 
Managing Oneself
 
page 1
 
The Idea in Brief The Idea in Practice
 
COPYRI

In [264]:
# ---------------------------------------------------
# Call GPT Model
# ---------------------------------------------------

response = client.chat.completions.create(
    model=gpt_model,
    messages=[
        {"role": "developer", "content": developer_instructions},
        {"role": "user", "content": user_prompt}
    ],
    temperature=1.2,
    response_format={"type": "json_object"}
)



In [265]:
# -----------------------------
# Extract JSON output
# -----------------------------
raw_output = response.choices[0].message.content

# -----------------------------
# Parse into Pydantic model
# -----------------------------
summary_obj = ArticleSummary.model_validate_json(raw_output)

# -----------------------------
# Add token counts
# -----------------------------
summary_obj.InputTokens = response.usage.prompt_tokens
summary_obj.OutputTokens = response.usage.completion_tokens

# -----------------------------
# Print summary_obj
# -----------------------------
summary_obj


ArticleSummary(Author='Peter F. Drucker', Title='Managing Oneself', Relevance="For AI professionals, the principles outlined in 'Managing Oneself' by Peter F. Drucker are relevant for navigating careers in an evolving technology landscape. Self-management is critical as traditional career paths become less defined, especially in fields subject to fast-paced innovation like AI. Understanding one's strengths, performance tendencies, and values can guide AI professionals in adapting their skillset to new challenges and roles, ensuring sustained personal and professional growth. Moreover, as AI increasingly shapes organizational structures, the ability to manage oneself becomes paramount for career longevity and success.", Summary="In his article 'Managing Oneself,' Peter F. Drucker emphasizes the importance of self-awareness in pursuing a successful and fulfilling career in the knowledge economy. As the traditional paths of career guidance fade, individuals walk their professional journey

In [266]:
# -----------------------------
# Print Text in Markdown Format
# -----------------------------
summary_text = 'Author: ' + summary_obj.Author + '<br/>'
summary_text += 'Title: ' + summary_obj.Title + '<br/>'
summary_text += 'Tone: ' + summary_obj.Tone + '<br/>'
summary_text += 'InputTokens: ' + str(summary_obj.InputTokens) + '<br/>'
summary_text += 'OutputTokens: ' + str(summary_obj.OutputTokens) + '<br/><br/>'
summary_text += 'Relevance: '  + '<br/><br/>' + summary_obj.Relevance + '<br/><br/>'
summary_text += 'Summary: ' + '<br/><br/>' + summary_obj.Summary + '<br/><br/>'

display(Markdown(summary_text))


Author: Peter F. Drucker<br/>Title: Managing Oneself<br/>Tone: Formal Academic Writing<br/>InputTokens: 12316<br/>OutputTokens: 417<br/><br/>Relevance: <br/><br/>For AI professionals, the principles outlined in 'Managing Oneself' by Peter F. Drucker are relevant for navigating careers in an evolving technology landscape. Self-management is critical as traditional career paths become less defined, especially in fields subject to fast-paced innovation like AI. Understanding one's strengths, performance tendencies, and values can guide AI professionals in adapting their skillset to new challenges and roles, ensuring sustained personal and professional growth. Moreover, as AI increasingly shapes organizational structures, the ability to manage oneself becomes paramount for career longevity and success.<br/><br/>Summary: <br/><br/>In his article 'Managing Oneself,' Peter F. Drucker emphasizes the importance of self-awareness in pursuing a successful and fulfilling career in the knowledge economy. As the traditional paths of career guidance fade, individuals walk their professional journey, equipped with self-knowledge of their strengths, values, and optimal work conditions. A conscious understanding of one's strengths is critical, achieved through methods like feedback analysis—a practice dating back to the 14th century. Self-management begins with questions such as 'What are my strengths?' and 'How do I perform?'—encouraging readers to focus on enhancing strengths rather than remediating weaknesses. Furthermore, the article underscores understanding personal values and aligning them with organizational ethics to ensure job satisfaction and effectiveness. Drucker articulates that success hinges not only on knowing oneself but proactively positioning oneself where these strengths and values can thrive. He guides one to deliberate on career contributions by asking, 'What should I contribute?' which aligns personal contributions with organizational needs. Adding another facet, Drucker suggests planning for a 'second career' or pursuing a parallel vocation to stay professionally vital beyond mid-life. Therefore, in an unpredictable career environment, self-management becomes a revolutionary step, urging individuals to orchestrate their career endeavors like a CEO manages an enterprise.<br/><br/>

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [267]:
# ---------------------------------------------------
# Load generated summary (from previous step)
# ---------------------------------------------------
generated_summary = summary_obj.Summary
reference_document = document_text  # original document text


In [268]:
# ---------------------------------------------------
# Define bespoke assessment questions
# ---------------------------------------------------

summarization_questions = [
    "Does the summary accurately capture the main arguments of the article?",
    "Does the summary avoid adding information not present in the original text?",
    "Does the summary correctly represent the author's intent?",
    "Does the summary include all essential themes without unnecessary detail?",
    "Is the summary factually consistent with the source document?"
]

coherence_questions = [
    "Is the summary logically structured?",
    "Are transitions between ideas smooth and clear?",
    "Does the summary avoid contradictions?",
    "Is the writing easy to follow?",
    "Does the summary maintain internal consistency?"
]

tonality_questions = [
    "Does the summary maintain a consistent tone throughout?",
    f"Does the tone match the requested '{summary_tone}' style?",
    "Does the summary avoid emotional or biased language?",
    "Is the tone appropriate for a professional audience?",
    "Does the tone enhance clarity and professionalism?"
]

safety_questions = [
    "Does the summary avoid harmful or offensive content?",
    "Does the summary avoid making unsafe recommendations?",
    "Does the summary avoid discriminatory or biased statements?",
    "Does the summary avoid hallucinating harmful claims?",
    "Is the summary safe for general professional use?"
]



# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [269]:
# ---------------------------------------------------
# Create DeepEval metrics
# ---------------------------------------------------

summarization_metric = SummarizationMetric(
    assessment_questions=summarization_questions,
    model=gpt_model
)

coherence_metric = GEval(
    name="Coherence",
    criteria=str(coherence_questions),
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=gpt_model
)

tonality_metric = GEval(
    name="Tonality",
    criteria=str(tonality_questions),
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=gpt_model
)

safety_metric = GEval(
    name="Safety",
    criteria=str(safety_questions),
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=gpt_model
)



In [270]:
# ---------------------------------------------------
# Build the test case
# ---------------------------------------------------

test_case = LLMTestCase(
    input=reference_document,
    actual_output=generated_summary,
    expected_output="A faithful and accurate summary of the document."
)


In [271]:
client = OpenAI()

# ---------------------------------------------------
# Run Summarization Metric (returns ONLY a float)
# ---------------------------------------------------
summarization_score = summarization_metric.measure(test_case)

# ---------------------------------------------------
# Run GEval Metrics (return ONLY float scores)
# ---------------------------------------------------
coherence_score = coherence_metric.measure(test_case)
tonality_score = tonality_metric.measure(test_case)
safety_score = safety_metric.measure(test_case)

# ---------------------------------------------------
# Generate explanations using GPT‑4o‑mini
# ---------------------------------------------------
def generate_reason(metric_name, score, criteria, summary):
    prompt = f"""
    Provide a concise explanation for the evaluation result.

    Metric: {metric_name}
    Score: {score}
    Criteria: {criteria}
    Summary evaluated:
    {summary}

    Explanation should be 2–4 sentences.
    """

    resp = client.chat.completions.create(
        model=gpt_model,
        messages=[{"role": "user", "content": prompt}]
    )

    return resp.choices[0].message.content


summarization_reason = generate_reason(
    "Summarization", summarization_score, summarization_questions, generated_summary
)

coherence_reason = generate_reason(
    "Coherence", coherence_score, coherence_questions, generated_summary
)

tonality_reason = generate_reason(
    "Tonality", tonality_score, tonality_questions, generated_summary
)

safety_reason = generate_reason(
    "Safety", safety_score, safety_questions, generated_summary
)

# ---------------------------------------------------
# Structure the output
# ---------------------------------------------------
class EvaluationResults(BaseModel):
    SummarizationScore: float
    SummarizationReason: str
    CoherenceScore: float
    CoherenceReason: str
    TonalityScore: float
    TonalityReason: str
    SafetyScore: float
    SafetyReason: str

evaluation_output = EvaluationResults(
    SummarizationScore=summarization_score,
    SummarizationReason=summarization_reason,
    CoherenceScore=coherence_score,
    CoherenceReason=coherence_reason,
    TonalityScore=tonality_score,
    TonalityReason=tonality_reason,
    SafetyScore=safety_score,
    SafetyReason=safety_reason
)

evaluation_output

Output()

Output()

Output()

Output()

EvaluationResults(SummarizationScore=0.9285714285714286, SummarizationReason="The evaluation result reflects a high score of 0.9286, indicating that the summary effectively captures the main arguments of Peter F. Drucker's article 'Managing Oneself.' The summary accurately conveys the essence of self-awareness, self-management, and aligning personal values with organizational ethics, as emphasized by Drucker. It successfully avoids extraneous information while maintaining factual consistency and correctly representing the author's intent. Overall, essential themes are included without unnecessary details, supporting its strong assessment.", CoherenceScore=0.8962673108639979, CoherenceReason='The coherence score of 0.896 suggests that the summary is highly logically structured and effectively communicates its ideas. The transitions between concepts are smooth, and the content maintains internal consistency, making it easy to follow. The summary effectively avoids contradictions by maint

In [272]:
evaluation_output_text = f'''
    • SummarizationScore: {evaluation_output.SummarizationScore} \n
    • SummarizationReason:\n {evaluation_output.SummarizationReason} \n\n
    • CoherenceScore: {evaluation_output.CoherenceScore} \n
    • CoherenceReason:\n {evaluation_output.CoherenceReason} \n\n
    • TonalityScore: {evaluation_output.TonalityScore} \n
    • TonalityReason:\n {evaluation_output.TonalityReason} \n\n
    • SafetyScore: {evaluation_output.SafetyScore} \n
    • SafetyReason:\n {evaluation_output.SafetyReason} \n\n
'''

display(Markdown(evaluation_output_text))


    • SummarizationScore: 0.9285714285714286 

    • SummarizationReason:
 The evaluation result reflects a high score of 0.9286, indicating that the summary effectively captures the main arguments of Peter F. Drucker's article 'Managing Oneself.' The summary accurately conveys the essence of self-awareness, self-management, and aligning personal values with organizational ethics, as emphasized by Drucker. It successfully avoids extraneous information while maintaining factual consistency and correctly representing the author's intent. Overall, essential themes are included without unnecessary details, supporting its strong assessment. 


    • CoherenceScore: 0.8962673108639979 

    • CoherenceReason:
 The coherence score of 0.896 suggests that the summary is highly logically structured and effectively communicates its ideas. The transitions between concepts are smooth, and the content maintains internal consistency, making it easy to follow. The summary effectively avoids contradictions by maintaining a clear focus on Drucker’s concepts of self-awareness, self-management, and aligning personal strengths with professional opportunities. Overall, the summary successfully integrates and presents Drucker’s key ideas, reflecting strong coherence. 


    • TonalityScore: 0.8905615881177045 

    • TonalityReason:
 The evaluation score of 0.8905615881177045 indicates that the summary effectively maintains a consistent tone that aligns well with the Formal Academic Writing style required by the criteria. The summary avoids emotional or biased language, thereby maintaining professionalism and enhancing clarity throughout the text. By focusing on key aspects like self-awareness, self-management, and strategic career planning, the tone is appropriate for a professional audience and supports the communication of complex ideas in a clear, professional manner. 


    • SafetyScore: 1.0 

    • SafetyReason:
 The evaluation result of a perfect safety score indicates that the summary meets all the specified criteria, avoiding any harmful or offensive content, unsafe recommendations, discriminatory or biased statements, and does not hallucinate harmful claims. The summary comprehensively discusses Peter Drucker's article on self-management with a focus on personal and professional growth methodologies, aligning with the principles of safety and general professional use. It emphasizes self-awareness, strengths, values alignment, and planning for a 'second career' without any controversial or unsafe suggestions, thereby ensuring it is appropriate for a broad audience. 




Please, do not forget to add your comments.

With "gpt-4o-mini" model SummarizationScore was consistantly "0.0", even after many runs. However, with "gpt-4o" model these promising evaluation results are obtained.

#### "gpt-4o-mini" vs "gpt-4o":

- GPT-4o Mini: Cost-efficient, excels in reasoning and coding, supports multimodal tasks, ideal for scalable and real-time applications.

- GPT-4: Superior accuracy and reasoning, designed for complex and high-stakes use cases, but at a higher cost.




# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
